In [ ]:
!pip install --upgrade google-generativeai
!pip install PyPDF2 python-dotenv
!pip install gradio
!pip install google-generativeai SpeechRecognition pydub threadpoolctl
!apt-get update
!apt-get install -y portaudio19-dev python3-dev
!pip install pyaudio --no-cache-dir
!apt-get install -y ffmpeg
!pip install moviepy

In [ ]:
import gradio as gr
import google.generativeai as genai
import speech_recognition as sr
import tempfile
import os
from moviepy.editor import VideoFileClip
from concurrent.futures import ThreadPoolExecutor

# ********** CONFIGURATION **********
GEMINI_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual API key
MODEL_NAME = "gemini-1.5-flash"  # Or another available model like gemini-1.0-pro

# Initialize Gemini
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel(
    MODEL_NAME,
    generation_config={
        "temperature": 0.3,
        "max_output_tokens": 1500,
        "top_p": 0.95
    }
)

executor = ThreadPoolExecutor(max_workers=4)
recognizer = sr.Recognizer()

SUPPORTED_AUDIO = ["wav", "mp3", "ogg", "flac"]
SUPPORTED_VIDEO = ["mp4", "mov", "avi", "mkv"]

custom_css = """
.header {
    text-align: center;
    padding: 20px;
    background: linear-gradient(45deg, #4299e1, #48bb78);
    border-radius: 10px;
    margin-bottom: 20px;
}
.card {
    padding: 20px;
    border-radius: 10px;
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
    margin-bottom: 20px;
    background: white;
}
.input-style {
    border: 2px solid #cbd5e0 !important;
    border-radius: 8px !important;
    padding: 15px !important;
    color: #000000 !important;
}
.markdown-output {
    padding: 20px;
    border-radius: 8px;
    background: #f7fafc;
    border: 2px solid #e2e8f0;
    color: #000000 !important;
}
body, label, h1, h2, h3, h4, h5, h6 {
    color: #000000 !important;
}
"""

def handle_api_errors(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except genai.types.BlockedPromptError as e:
            return f"⚠️ Content blocked: {e}"
        except Exception as e:
            return f"❌ Error: {str(e)}"
    return wrapper

def optimize_prompt(text, max_length):
    return text.strip().replace('\n', ' ')[:max_length]

@handle_api_errors
def analyze_resume(resume_text, job_desc):
    if not (resume_text.strip() and job_desc.strip()):
        return "❌ Please provide both resume and job description"

    cleaned_resume = optimize_prompt(resume_text, 2000)
    cleaned_job_desc = optimize_prompt(job_desc, 1000)

    prompt = f"""**Resume Analysis**
Resume: {cleaned_resume}
Job: {cleaned_job_desc}

Provide concise markdown analysis with:
1. 🎯 Skills Match Percentage
2. 🔍 Missing Keywords
3. 💡 Top 2 Improvement Suggestions
4. ✅ Key Strengths"""

    return executor.submit(model.generate_content, prompt).result().text

def process_media(file_path):
    try:
        ext = os.path.splitext(file_path)[1][1:].lower()
        audio_path = None

        if ext in SUPPORTED_VIDEO:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                VideoFileClip(file_path).audio.write_audiofile(
                    tmp.name,
                    codec='pcm_s16le',
                    ffmpeg_params=['-loglevel', 'quiet']
                )
                audio_path = tmp.name

        elif ext in SUPPORTED_AUDIO:
            audio_path = file_path

        if audio_path:
            with sr.AudioFile(audio_path) as source:
                audio = recognizer.record(source)
                return recognizer.recognize_google(audio)
        return f"Unsupported format: {ext}"

    except Exception as e:
        return f"❌ Media error: {str(e)}"
    finally:
        if 'audio_path' in locals() and audio_path and os.path.exists(audio_path):
            os.unlink(audio_path)

@handle_api_errors
def process_meeting(input_data, media_file):
    text_chunks = []

    if input_data.strip():
        text_chunks.append(optimize_prompt(input_data, 1500))

    if media_file:
        media_text = process_media(media_file)
        if '❌' not in media_text:
            text_chunks.append(optimize_prompt(media_text, 1500))

    if not text_chunks:
        return "❌ No input provided"

    full_text = "\n".join(text_chunks)[:3000]

    prompt = f"""**Meeting Summary**
{full_text}

Create concise markdown summary with:
- 📌 Key Decisions
- ✅ Action Items (owners)
- 💬 Main Points
- 🚀 Next Steps"""

    return executor.submit(model.generate_content, prompt).result().text

@handle_api_errors
def ai_tutor(question, subject):
    if not question.strip():
        return "❌ Please enter a question"

    prompt = f"""**Tutor: {subject}**
{optimize_prompt(question, 500)}

Explain with:
1. Core Principles
2. Key Formulas
3. Real-world Examples
4. Common Mistakes"""

    return executor.submit(model.generate_content, prompt).result().text

@handle_api_errors
def code_assistant(code, task):
    if not code.strip():
        return "❌ Please enter some code"

    prompt = f"""**Code {task}**
{optimize_prompt(code, 2000)}

Provide:
1. Optimized Solution
2. Step Explanation
3. Alternative Approaches
4. Performance Tips"""

    return executor.submit(model.generate_content, prompt).result().text

# Gradio UI
with gr.Blocks(title="AI Productivity Suite", css=custom_css, theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    <div class="header">
        <h1>🚀 AI Productivity Suite</h1>
        <p>Supports Video/Audio Meetings</p>
    </div>
    """)

    with gr.Tabs():
        with gr.Tab("📈 Productivity", elem_classes=["card"]):
            with gr.Row():
                with gr.Column(scale=2):
                    gr.Markdown("### 📄 Resume Analyzer")
                    with gr.Group(elem_classes=["card"]):
                        resume_input = gr.Textbox(label="Paste Resume", lines=10, elem_classes=["input-style"])
                        job_desc_input = gr.Textbox(label="Paste Job Description", lines=5, elem_classes=["input-style"])
                        analyze_btn = gr.Button("Analyze", variant="primary")
                    resume_output = gr.Markdown(elem_classes=["markdown-output"])

                with gr.Column(scale=1):
                    gr.Markdown("### 🎥 Meeting Manager")
                    with gr.Group(elem_classes=["card"]):
                        meeting_input = gr.Textbox(label="Text Input", lines=10, elem_classes=["input-style"])
                        media_input = gr.File(label="Upload Audio/Video", file_types=SUPPORTED_AUDIO + SUPPORTED_VIDEO)
                        summarize_btn = gr.Button("Summarize", variant="primary")
                    meeting_output = gr.Markdown(elem_classes=["markdown-output"])

        with gr.Tab("⚙ Engineering", elem_classes=["card"]):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 🎓 Smart Tutor")
                    with gr.Group(elem_classes=["card"]):
                        subject = gr.Dropdown(["Programming", "Electronics", "Mechanics"], label="Subject")
                        question = gr.Textbox(label="Question", elem_classes=["input-style"])
                        tutor_btn = gr.Button("Explain", variant="primary")
                    tutor_output = gr.Markdown(elem_classes=["markdown-output"])

                with gr.Column():
                    gr.Markdown("### 💻 Code Assistant")
                    with gr.Group(elem_classes=["card"]):
                        code_input = gr.Code(label="Code", language="python", lines=15)
                        task = gr.Dropdown(["Debug", "Optimize", "Explain"], label="Task")
                        code_btn = gr.Button("Analyze", variant="primary")
                    code_output = gr.Markdown(elem_classes=["markdown-output"])

    analyze_btn.click(analyze_resume, [resume_input, job_desc_input], resume_output)
    summarize_btn.click(process_meeting, [meeting_input, media_input], meeting_output)
    tutor_btn.click(ai_tutor, [question, subject], tutor_output)
    code_btn.click(code_assistant, [code_input, task], code_output)

if __name__ == "__main__":
    app.launch()
